In [ ]:
import requests
import lxml.html
from collections import defaultdict
import pandas as pd

## State Court Data Ingestion
Exploration of data sources for state courts for ingestion into Courts table. Working code now lives in ingest_courts_data.py.

Discussion: I think goal of functions in ingestion should be to produce an output which can be easily loaded into Django - not totally sure what the best format for that is, I think bulk create method takes a dict or we could go data -> pandas dataframe -> pandas to django convenience methods -> django. Should decide on a common output for ingestion functions.

### NCSC State Court Info
Scrape data on court selection methods, term lengths, etc for state courts from archived NCSC site: http://web.archive.org/web/20151022190427/http://www.judicialselection.us/judicial_selection/methods/selection_of_judges.cfm?state=

In [ ]:
# request and parse html
archive_ncsc_url = "http://web.archive.org/web/20151022190427/http://www.judicialselection.us/judicial_selection/methods/selection_of_judges.cfm?state="
r = requests.get(archive_ncsc_url)
response = lxml.html.fromstring(r.text)

In [ ]:
# parse html for state court info
state_data_dict = defaultdict(lambda: defaultdict(dict))
other_thing = []

for item in response.xpath('//div[@id="content"]/*'):
    if item.tag in ["h2", "h3", "p"]:
        continue
    # update the state
    elif item.tag == "div" and "yellow_box" in item.classes:
        state_str = item.xpath("h4/text()")[0]
    # get data for one state
    elif item.tag == "table":
        state_data_str = item.text_content()
        state_data_list = [
            item for item in state_data_str.replace("\t", "").split("\n") if item != ""
        ]
        stat_key = state_data_list[0]

        i = 1
        while i < len(state_data_list) - 1:
            court = state_data_list[i]
            stat_value = state_data_list[i + 1]
            state_data_dict[state_str][court][stat_key] = stat_value
            i += 2

state_data_dict

defaultdict(<function <lambda> at 0x0000027B1E16AFC0>, {'Alabama': defaultdict(<class 'dict'>, {'Supreme Court:': {'Number of Judgeships': '9', 'Number of Districts/Circuits': '--', 'Geographic Basis for Selection': 'statewide', 'Method of Selection (full term)': 'partisan election', 'Length of Term': '6 yrs', 'Method of Retention': 'reelection', 'Length of Subsequent Terms': '6 yrs', 'Method of Filling Interim Vacancies': 'gubernatorial appointment', 'When Interim Judges Stand for Election/Appointment': 'next general election after 1 yr in office', 'Selection of Chief Judge/Justice': 'popular election', 'Term of Office for Chief Judge/Justice': '6 yrs', 'Qualifications': 'licensed to practice law 10 yrs; 1 yr resident; maximum age of 70'}, 'Court of Criminal Appeals:': {'Number of Judgeships': '5', 'Number of Districts/Circuits': '--', 'Geographic Basis for Selection': 'statewide', 'Method of Selection (full term)': 'partisan election', 'Length of Term': '6 yrs', 'Method of Retention'

In [ ]:
# variable names improved in real code
# reformat to flat dictionaries and convert to dataframe
things = []
for state, court_dict in state_data_dict.items():
    for court, info in court_dict.items():
        flat = {"State": state, "Court": court, **info}
        things.append(flat)
df = pd.DataFrame(things)
df

,State,Court,Number of Judgeships,Number of Districts/Circuits,Geographic Basis for Selection,Method of Selection (full term),Length of Term,Method of Retention,Length of Subsequent Terms,Method of Filling Interim Vacancies,When Interim Judges Stand for Election/Appointment,Selection of Chief Judge/Justice,Term of Office for Chief Judge/Justice,Qualifications
0,Alabama,Supreme Court:,9,--,statewide,partisan election,6 yrs,reelection,6 yrs,gubernatorial appointment,next general election after 1 yr in office,popular election,6 yrs,licensed to practice law 10 yrs; 1 yr resident...
1,Alabama,Court of Criminal Appeals:,5,--,statewide,partisan election,6 yrs,reelection,6 yrs,gubernatorial appointment,next general election after 1 yr in office,peer vote,indefinite,licensed to practice law 10 yrs; 1 yr resident...
2,Alabama,Court of Civil Appeals:,5,--,statewide,partisan election,6 yrs,reelection,6 yrs,gubernatorial appointment,next general election after 1 yr in office,seniority,indefinite,licensed to practice law 10 yrs; 1 yr resident...
3,Alabama,Circuit Court:,144,41,circuit,partisan election,6 yrs,reelection,6 yrs,gubernatorial appointment*,next general election after 1 yr in office,peer vote,3 yrs,licensed to practice law 5 yrs; 1 yr resident ...
4,Alaska,Supreme Court:,5,--,statewide,gubernatorial appointment from nominating comm...,at least 3 yrs,retention election,10 yrs,gubernatorial appointment from nominating comm...,first general election more than 3 yrs after a...,peer vote,3 yrs,U.S. citizen; state resident 5 yrs; licensed t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,Wisconsin,Supreme Court:,7,--,statewide,nonpartisan election*,10 yrs,reelection,10 yrs,gubernatorial appointment**,next spring in which no other justice is to be...,seniority,indefinite,qualified elector of the state; licensed to pr...
151,Wisconsin,Court of Appeals:,16,4,district,nonpartisan election*,6 yrs,reelection,6 yrs,gubernatorial appointment**,next spring election in which no other judge f...,supreme court appoints,3 yrs,qualified elector of the state; licensed to pr...
152,Wisconsin,Circuit Court:,241,69,circuit,nonpartisan election*,6 yrs,reelection,6 yrs,gubernatorial appointment**,next spring election (if the vacancy occurs be...,supreme court appoints,2 yrs,qualified elector of circuit; licensed to prac...
153,Wyoming,Supreme Court:,5,--,statewide,gubernatorial appointment from nominating comm...,at least one year,retention election,8 yrs,gubernatorial appointment from nominating comm...,next general election 1 yr after appointment; ...,peer vote,4 yrs,U.S. citizen; state resident 3 yrs; 9 yrs lega...


In [77]:
selection_methods = (
    df["Method of Selection (full term)"].str.replace("*", "").str.replace("+", "").unique()
)
selection_methods

<StringArray>
[                                                                                      'partisan election',
                                                    'gubernatorial appointment from nominating commission',
     'partisan primary; nonpartisan general election/gubernatorial appointment from nominating commission',
                                                                                    'nonpartisan election',
                           'gubernatorial appointment;confirmation by commission on judicial appointments',
                    'gubernatorial nomination from judicial selection commission; legislative appointment',
                       'gubernatorial appointment from judicial nominating commission with senate consent',
                  'presidential appointment from judicial nomination commission, with senate confirmation',
                           'gubernatorial appointment from nominating commission with senate confirmation',
              

### CourtListener
Get list of courts from CourtListener's dedicated courts API: https://www.courtlistener.com/api/rest/v4/courts/

API token required! Just need to sign in on CourtListener site and find token under developer tools.

In [ ]:
base_url = "https://www.courtlistener.com/"
courts_url = "api/rest/v4/courts/"
API_token = "REPLACE WITH YOUR API TOKEN"

In [13]:
courts_request = requests.get(
    base_url + courts_url, headers={"Authorization": f"Token {API_token}"}
)
courts_request.status_code

200

In [ ]:
courts_results = []
page_url = base_url + courts_url
while page_url is not None:
    r = requests.get(page_url, headers={"Authorization": f"Token {API_token}"})
    print(page_url, r.status_code)
    response = r.json()
    results = response["results"]
    courts_results.extend(results)
    page_url = response["next"]

In [ ]:
courtlistener_df = pd.DataFrame(courts_results)
courtlistener_df

Lots of the variables are kind of confusing; some jurisdictions mislabeled etc. Their state id decisions are crazy (Alabama is ala, Alaska is alaska, etc. wtf)

In [ ]:
# filter to jurisdiction "S" for statewide courts and non-NaN start date,
# NaN end date to get state supreme courts only.
cl_ssc_df = courtlistener_df[
    (courtlistener_df["jurisdiction"] == "S")
    & (courtlistener_df["start_date"].notna())
    & (courtlistener_df["end_date"].isna())
]
cl_ssc_df

,resource_uri,id,pacer_court_id,pacer_has_rss_feed,pacer_rss_entry_types,date_last_pacer_contact,fjc_court_id,date_modified,in_use,has_opinion_scraper,...,position,citation_string,short_name,full_name,url,start_date,end_date,jurisdiction,parent_court,appeals_to
393,https://www.courtlistener.com/api/rest/v4/cour...,ala,NaN,None,,None,,2023-01-06T11:30:36.039561-08:00,True,True,...,305.000,Ala.,Supreme Court of Alabama,Supreme Court of Alabama,http://judicial.alabama.gov/supreme.cfm,1819-01-01,NaN,S,NaN,[]
397,https://www.courtlistener.com/api/rest/v4/cour...,alaska,NaN,None,,None,,2013-08-14T09:46:30-07:00,True,True,...,320.000,Alaska,Alaska Supreme Court,Alaska Supreme Court,http://courts.alaska.gov/sp.htm,1959-01-03,NaN,S,NaN,[]
399,https://www.courtlistener.com/api/rest/v4/cour...,ariz,NaN,None,,None,,2013-08-14T09:46:30-07:00,True,True,...,322.000,Ariz.,Arizona Supreme Court,Arizona Supreme Court,http://www.azcourts.gov/azsupremecourt.aspx,1912-01-01,NaN,S,NaN,[]
403,https://www.courtlistener.com/api/rest/v4/cour...,ark,NaN,None,,None,,2013-09-08T18:32:48-07:00,True,True,...,330.000,Ark.,Supreme Court of Arkansas,Supreme Court of Arkansas,https://courts.arkansas.gov/courts/supreme-court,1841-01-01,NaN,S,NaN,[]
407,https://www.courtlistener.com/api/rest/v4/cour...,cal,NaN,None,,None,,2013-08-14T09:46:30-07:00,True,True,...,350.000,Cal.,California Supreme Court,California Supreme Court,http://www.courts.ca.gov/supremecourt.htm,1849-01-01,NaN,S,NaN,[]
571,https://www.courtlistener.com/api/rest/v4/cour...,colo,NaN,None,,None,,2014-07-11T13:49:42.750000-07:00,True,True,...,350.900,Colo.,Supreme Court of Colorado,Supreme Court of Colorado,http://www.coloradosupremecourt.com/,1876-08-01,NaN,S,NaN,[]
590,https://www.courtlistener.com/api/rest/v4/cour...,conn,NaN,None,,None,,2014-07-17T14:35:19.385000-07:00,True,True,...,351.500,Conn.,Supreme Court of Connecticut,Supreme Court of Connecticut,http://www.jud.state.ct.us/external/supapp/,1784-01-01,NaN,S,NaN,[]
594,https://www.courtlistener.com/api/rest/v4/cour...,del,NaN,None,,None,,2016-09-08T13:39:19.377323-07:00,True,True,...,352.000,Del.,Supreme Court of Delaware,Supreme Court of Delaware,http://courts.delaware.gov/supreme/,1832-01-01,NaN,S,NaN,[]
606,https://www.courtlistener.com/api/rest/v4/cour...,fla,NaN,None,,None,,2014-08-01T09:49:22.115000-07:00,True,True,...,354.000,Fla.,Supreme Court of Florida,Supreme Court of Florida,http://www.floridasupremecourt.org/,1845-01-01,NaN,S,NaN,[]
813,https://www.courtlistener.com/api/rest/v4/cour...,ga,NaN,None,,None,,2014-08-01T10:11:53.684000-07:00,True,True,...,355.500,Ga.,Supreme Court of Georgia,Supreme Court of Georgia,http://www.gasupreme.us/,1841-01-01,NaN,S,NaN,[]


In [ ]:
# save to csv file
# courtlistener_df.to_csv("courts_courtlistener.csv")